# Virtual Try-On on VITON-HD

- model: CatVTON (diffusion, pretrained on VITON-HD at 512×384)
- data: VITON-HD test split (2,032 pairs)
- tests: paired (SSIM, LPIPS) and unpaired (FID, KID)
- failure analysis: worst and best results by LPIPS
- demo: Gradio app with a public link

**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all.

## 1. Settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

N_PAIRED = 200      # test people wearing their own shirt
N_UNPAIRED = 200    # test people wearing a different shirt
STEPS = 30          # diffusion steps (catvton default is 50)
GUIDANCE = 2.5
H, W = 512, 384
SEED = 42
SAVE_TO_DRIVE = True  # keeps outputs if colab disconnects

## 2. Drive and output folder

In [ ]:
import os
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/viton/results'
else:
    OUT = '/content/results'
for sub in ['paired', 'unpaired', 'grids']:
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)
print('outputs ->', OUT)

## 3. Install and get CatVTON code

In [ ]:
!pip -q install diffusers accelerate transformers peft torchmetrics torch-fidelity lpips gdown gradio
!rm -rf /content/CatVTON
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON /content/CatVTON

import sys
sys.path.insert(0, '/content/CatVTON')
os.chdir('/content/CatVTON')
print(sorted(os.listdir('.')))

## 4. Get VITON-HD

- official link from the VITON-HD repo (google drive)
- if gdown fails with a quota error: open the link in a browser, add it to your drive, put the zip at `MyDrive/viton/zalando-hd-resized.zip`, and run this cell again

In [ ]:
import glob, shutil
ZIP = '/content/viton.zip'
DRIVE_ZIP = '/content/drive/MyDrive/viton/zalando-hd-resized.zip'

if not os.path.exists(ZIP):
    if os.path.exists(DRIVE_ZIP):
        shutil.copy(DRIVE_ZIP, ZIP)
    else:
        !gdown 1tLx8LRp-sxDp0EcYmYoV_vXdSc-jJ79w -O {ZIP}

size_gb = os.path.getsize(ZIP) / 1e9 if os.path.exists(ZIP) else 0
print(f'zip: {size_gb:.2f} GB')
assert size_gb > 1, 'download failed, see the note above this cell'

# test split only
!unzip -q -o {ZIP} -x '*train/*' -d /content/vitonhd

TEST = os.path.dirname(glob.glob('/content/vitonhd/**/test/image', recursive=True)[0])
PAIRS = glob.glob('/content/vitonhd/**/test_pairs.txt', recursive=True)[0]
print('test folder:', TEST)
print('subfolders:', sorted(os.listdir(TEST)))
print('pairs file:', PAIRS)
!head -3 {PAIRS}

## 5. Masks

- the mask marks the area the model repaints (old shirt and arms)
- built from the dataset's `agnostic-v3.2` images (gray area), or from the body-part map if that folder is missing

In [ ]:
import numpy as np, cv2
from PIL import Image
import matplotlib.pyplot as plt

def mask_from_agnostic(name):
    img = np.asarray(Image.open(f'{TEST}/image/{name}').convert('RGB')).astype(int)
    agn = np.asarray(Image.open(f'{TEST}/agnostic-v3.2/{name}').convert('RGB')).astype(int)
    m = (np.abs(img - agn).sum(2) > 40).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    m = cv2.dilate(m, np.ones((9, 9), np.uint8))
    return Image.fromarray(m * 255)

def mask_from_parse(name):
    p = np.asarray(Image.open(f'{TEST}/image-parse-v3/{name.replace(".jpg", ".png")}'))
    m = np.isin(p, [5, 6, 7, 14, 15]).astype(np.uint8)  # upper clothes, dress, coat, arms
    m = cv2.dilate(m, np.ones((25, 25), np.uint8))
    return Image.fromarray(m * 255)

USE_AGNOSTIC = os.path.isdir(f'{TEST}/agnostic-v3.2')
get_mask = mask_from_agnostic if USE_AGNOSTIC else mask_from_parse
print('mask source:', 'agnostic-v3.2' if USE_AGNOSTIC else 'image-parse-v3')

names = sorted(os.listdir(f'{TEST}/image'))
fig, ax = plt.subplots(2, 4, figsize=(12, 8))
for i, n in enumerate(names[:4]):
    ax[0, i].imshow(Image.open(f'{TEST}/image/{n}')); ax[0, i].set_title(n); ax[0, i].axis('off')
    ax[1, i].imshow(get_mask(n), cmap='gray'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 6. Load CatVTON

In [ ]:
import torch, types, functools, tqdm as _tqdm
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
import model.pipeline as mp
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

mp.tqdm = types.SimpleNamespace(tqdm=functools.partial(_tqdm.tqdm, disable=True))  # quiet per-step bars

REPO = snapshot_download('zhengchong/CatVTON', allow_patterns=['vitonhd-16k-512/*', 'DensePose/*', 'SCHP/*'])
pipe = CatVTONPipeline(
    base_ckpt='booksforcharlie/stable-diffusion-inpainting',
    attn_ckpt=REPO,
    attn_ckpt_version='vitonhd',
    weight_dtype=torch.float16,
    device='cuda',
    skip_safety_check=True,
    use_tf32=False,
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)

def tryon(person, cloth, mask, steps=STEPS, seed=SEED):
    person = resize_and_crop(person.convert('RGB'), (W, H))
    mask = resize_and_crop(mask.convert('L'), (W, H))
    cloth = resize_and_padding(cloth.convert('RGB'), (W, H))
    mask = mask_processor.blur(mask, blur_factor=9)
    g = torch.Generator(device='cuda').manual_seed(int(seed))
    out = pipe(image=person, condition_image=cloth, mask=mask, num_inference_steps=int(steps),
               guidance_scale=GUIDANCE, height=H, width=W, generator=g)[0]
    return out, person, cloth, mask

print('gpu memory used: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

## 7. Run try-on

- paired: each person gets their own shirt, so the real photo is the answer
- unpaired: shirts from `test_pairs.txt` (person, cloth)
- already finished images are skipped, so after a disconnect just run again

In [ ]:
import random, time

paired = random.Random(0).sample(names, N_PAIRED)
with open(PAIRS) as f:
    pair_lines = [l.split() for l in f if l.strip()]
unpaired = random.Random(1).sample(pair_lines, N_UNPAIRED)

jobs = [('paired', n, n) for n in paired] + [('unpaired', p, c) for p, c in unpaired]
todo = [j for j in jobs if not os.path.exists(f'{OUT}/{j[0]}/{j[1].replace(".jpg", "")}__{j[2].replace(".jpg", "")}.png')]
print(f'{len(jobs)} jobs, {len(todo)} left')

t0 = time.time()
for i, (kind, p, c) in enumerate(todo):
    person = Image.open(f'{TEST}/image/{p}')
    cloth = Image.open(f'{TEST}/cloth/{c}')
    out, *_ = tryon(person, cloth, get_mask(p))
    out.save(f'{OUT}/{kind}/{p.replace(".jpg", "")}__{c.replace(".jpg", "")}.png')
    if i == 2 or (i + 1) % 25 == 0:
        per = (time.time() - t0) / (i + 1)
        print(f'{i + 1}/{len(todo)}  {per:.1f} s/img  ~{per * (len(todo) - i - 1) / 60:.0f} min left')
print('done in %.1f min' % ((time.time() - t0) / 60))

## 8. Quick look

In [ ]:
def show_row(ax, imgs, titles):
    for a, im, t in zip(ax, imgs, titles):
        a.imshow(im, cmap='gray'); a.set_title(t, fontsize=9); a.axis('off')

files = sorted(os.listdir(f'{OUT}/unpaired'))[:6]
fig, ax = plt.subplots(len(files), 3, figsize=(8, 3.6 * len(files)))
for r, f in enumerate(files):
    p, c = f[:-4].split('__')
    show_row(ax[r], [resize_and_crop(Image.open(f'{TEST}/image/{p}.jpg'), (W, H)),
                     resize_and_padding(Image.open(f'{TEST}/cloth/{c}.jpg').convert('RGB'), (W, H)),
                     Image.open(f'{OUT}/unpaired/{f}')], ['person', 'shirt', 'try-on'])
plt.tight_layout(); plt.savefig(f'{OUT}/grids/unpaired_examples.png', dpi=110); plt.show()

## 9. Metrics

- SSIM: higher is better (paired)
- LPIPS: lower is better (paired)
- FID, KID: lower is better (unpaired, against all 2,032 real test photos)
- FID with only a few hundred images reads high; KID is more stable on small sets

In [ ]:
import pandas as pd, json, torch
import torchvision.transforms.functional as TF
from torchmetrics.functional.image import structural_similarity_index_measure as ssim_fn
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

dev = 'cuda'
def to_t(im):
    return TF.to_tensor(im.convert('RGB')).unsqueeze(0).to(dev)

lpips = LearnedPerceptualImagePatchSimilarity(net_type='alex', normalize=True).to(dev)
rows = []
for f in sorted(os.listdir(f'{OUT}/paired')):
    p = f.split('__')[0]
    fake = to_t(Image.open(f'{OUT}/paired/{f}'))
    real = to_t(resize_and_crop(Image.open(f'{TEST}/image/{p}.jpg'), (W, H)))
    rows.append({'person': p, 'ssim': ssim_fn(fake, real, data_range=1.0).item(),
                 'lpips': lpips(fake, real).item()})
per_image = pd.DataFrame(rows)
per_image.to_csv(f'{OUT}/paired_per_image.csv', index=False)

fid = FrechetInceptionDistance(feature=2048, normalize=True).to(dev)
fake_files = sorted(os.listdir(f'{OUT}/unpaired'))
kid = KernelInceptionDistance(subset_size=min(100, len(fake_files)), normalize=True).to(dev)

def batches(paths, load, bs=32):
    for i in range(0, len(paths), bs):
        yield torch.cat([to_t(load(x)) for x in paths[i:i + bs]])

for b in batches(names, lambda n: resize_and_crop(Image.open(f'{TEST}/image/{n}'), (W, H))):
    fid.update(b, real=True); kid.update(b, real=True)
for b in batches(fake_files, lambda f: Image.open(f'{OUT}/unpaired/{f}')):
    fid.update(b, real=False); kid.update(b, real=False)
kid_mean, kid_std = kid.compute()

metrics = {
    'model': 'CatVTON vitonhd-16k-512', 'size': f'{W}x{H}', 'steps': STEPS, 'guidance': GUIDANCE,
    'n_paired': len(per_image), 'n_unpaired': len(fake_files), 'n_real': len(names),
    'ssim': round(per_image.ssim.mean(), 4), 'lpips': round(per_image.lpips.mean(), 4),
    'fid': round(fid.compute().item(), 2), 'kid': round(kid_mean.item(), 4), 'kid_std': round(kid_std.item(), 4),
}
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=2)
pd.Series(metrics)

## 10. Failure analysis

- worst and best 8 paired results by LPIPS
- columns: person, shirt, mask, try-on, real photo
- look for: logos and text, stripes, crossed arms, hair over shoulders, sleeve length changes

In [ ]:
def grid(df, title, path):
    fig, ax = plt.subplots(len(df), 5, figsize=(13, 3.6 * len(df)))
    for r, (_, row) in enumerate(df.iterrows()):
        p = row.person
        person = resize_and_crop(Image.open(f'{TEST}/image/{p}.jpg'), (W, H))
        show_row(ax[r], [person,
                         resize_and_padding(Image.open(f'{TEST}/cloth/{p}.jpg').convert('RGB'), (W, H)),
                         resize_and_crop(get_mask(f'{p}.jpg'), (W, H)),
                         Image.open(f'{OUT}/paired/{p}__{p}.png'),
                         person],
                 ['person', 'shirt', 'mask', f'try-on  lpips {row.lpips:.3f}', 'real'])
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=100); plt.show()

grid(per_image.nlargest(8, 'lpips'), 'worst 8 by LPIPS', f'{OUT}/grids/worst8.png')
grid(per_image.nsmallest(8, 'lpips'), 'best 8 by LPIPS', f'{OUT}/grids/best8.png')

## 11. Optional: use your own person photo in the demo

- needs detectron2 (for DensePose), builds in about 5–10 min
- skip this cell and the demo still works with test-set people

In [ ]:
import torch
torch.jit.script = lambda f: f  # same patch catvton uses before importing detectron2
!pip -q install fvcore iopath pycocotools omegaconf hydra-core yacs termcolor tabulate cloudpickle av
!pip -q install --no-build-isolation --no-deps 'git+https://github.com/facebookresearch/detectron2.git'

from model.cloth_masker import AutoMasker
automasker = AutoMasker(densepose_ckpt=os.path.join(REPO, 'DensePose'),
                        schp_ckpt=os.path.join(REPO, 'SCHP'), device='cuda')
print('automasker ready')

## 12. Demo

- pick a test person, or upload your own (needs step 11)
- upload any shirt photo, plain background works best
- open the `gradio.live` link it prints

In [ ]:
import gradio as gr
if 'automasker' not in globals():
    automasker = None

demo_people = paired[:30]

def run(choice, upload, shirt, steps, seed):
    if shirt is None:
        raise gr.Error('upload a shirt photo')
    if upload is not None:
        if automasker is None:
            raise gr.Error('run step 11 first to use your own photo, or clear it and pick a test person')
        person = resize_and_crop(upload.convert('RGB'), (W, H))
        mask = automasker(person, 'upper')['mask']
    else:
        person = Image.open(f'{TEST}/image/{choice}')
        mask = get_mask(choice)
    out, person, _, mask = tryon(person, shirt, mask, steps, seed)
    shown = np.asarray(person).copy()
    shown[np.asarray(mask.convert('L')) > 127] = 128
    return Image.fromarray(shown), out

with gr.Blocks(title='Virtual Try-On') as demo:
    gr.Markdown('## Virtual Try-On · CatVTON on VITON-HD')
    with gr.Row():
        with gr.Column():
            choice = gr.Dropdown(demo_people, value=demo_people[0], label='test person')
            upload = gr.Image(type='pil', label='or your own person photo (optional)')
            shirt = gr.Image(type='pil', label='shirt photo')
            steps = gr.Slider(10, 50, value=STEPS, step=5, label='steps')
            seed = gr.Number(value=SEED, label='seed')
            go = gr.Button('Try on', variant='primary')
        with gr.Column():
            masked = gr.Image(label='area to repaint')
            result = gr.Image(label='result')
    gr.Examples([[demo_people[0], None, f'{TEST}/cloth/{c}', STEPS, SEED] for _, c in unpaired[:4]],
                [choice, upload, shirt, steps, seed])
    go.click(run, [choice, upload, shirt, steps, seed], [masked, result])

demo.launch(share=True, debug=False)

## 13. Results summary

In [ ]:
print(json.dumps(metrics, indent=2))
print('saved in', OUT)
!ls {OUT} {OUT}/grids